# FMP Crypto

Read-first examples for FMP-backed crypto OHLCV data.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

CRYPTO_SYMBOL = "BTCUSD"

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

In [3]:
crypto_routes = {CRYPTO_PRICES_SECTION: CRYPTO_PRICES_LIBRARY}
route_table(crypto_routes)

,section,openbb_route,provider_library
0,crypto_prices,crypto.price.historical,fmp_crypto_prices


In [4]:
pd.Series(DEFAULT_CRYPTO_SYMBOLS, name="default_crypto_symbols")

0     BTCUSD
1     ETHUSD
2     SOLUSD
3     XRPUSD
4     ADAUSD
5    DOGEUSD
6     BNBUSD
7     LTCUSD
Name: default_crypto_symbols, dtype: object

In [5]:
if RUN_REFRESH:
    warehouse.market_prices.refresh(CRYPTO_SYMBOL, section=CRYPTO_PRICES_SECTION, provider=PROVIDER)

state_table(CRYPTO_SYMBOL, [CRYPTO_PRICES_SECTION])

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,BTCUSD,crypto_prices,fmp,2012-10-05,2026-06-20,5000,8,"(close, fmp__change, fmp__change_percent, fmp_...",2026-06-20T16:09:49.096621+00:00


In [6]:
preview(warehouse.market_prices.read(CRYPTO_SYMBOL, section=CRYPTO_PRICES_SECTION, provider=PROVIDER))

,open,high,low,close,volume,fmp__vwap,fmp__change,fmp__change_percent
date,,,,,,,,
2026-06-16,66276.790,66944.00,65300.030,65616.64,2.509124e+10,66034.3650,-660.15,-0.009961
2026-06-17,65616.640,66384.01,63851.020,64446.04,3.168786e+10,65074.4275,-1170.60,-0.017840
2026-06-18,64446.050,64752.51,62159.760,62879.01,3.065944e+10,63559.3325,-1567.04,-0.024316
2026-06-19,62879.010,63595.93,62246.240,63475.20,2.245724e+10,63049.0950,596.19,0.009482
2026-06-20,63484.438,64304.72,63182.547,64081.57,1.883870e+10,63856.2800,597.13,0.009406


<!-- quant-trader-eda -->
## Quant Trader EDA

Crypto checks: return/risk profile, tail moves, rolling volatility, and simple momentum state.

In [7]:
def price_eda(frame: pd.DataFrame, annualization: int = 365) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [8]:
crypto_prices = warehouse.market_prices.read(CRYPTO_SYMBOL, section=CRYPTO_PRICES_SECTION, provider=PROVIDER)
crypto_summary, crypto_diagnostics = price_eda(crypto_prices, annualization=365)
crypto_summary

,observations,start,end,total_return,annualized_return,annualized_volatility,sharpe_0_rf,max_drawdown,best_day,worst_day
0,4999,2012-10-05,2026-06-20,5049.565101,1.496179,0.765904,1.195853,-0.852983,0.486789,-0.388124


In [9]:
if crypto_prices.empty or "close" not in crypto_prices.columns:
    pd.DataFrame()
else:
    close = pd.to_numeric(crypto_prices["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    pd.DataFrame({
        "return_quantile": returns.quantile([0.01, 0.05, 0.50, 0.95, 0.99]),
        "abs_return_quantile": returns.abs().quantile([0.50, 0.90, 0.95, 0.99, 1.00]),
    })

In [10]:
if crypto_prices.empty or "close" not in crypto_prices.columns:
    pd.DataFrame()
else:
    close = pd.to_numeric(crypto_prices["close"], errors="coerce").dropna()
    momentum = pd.DataFrame({
        "close": close,
        "sma_20": close.rolling(20).mean(),
        "sma_100": close.rolling(100).mean(),
        "distance_to_100d_sma": close / close.rolling(100).mean() - 1,
    })
    preview(momentum, rows=10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

Crypto EDA should stress-test trend following, volatility targeting, and tail risk because uncompensated drawdowns dominate naive exposure.

In [11]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 365) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [12]:
crypto_prices = warehouse.market_prices.read(CRYPTO_SYMBOL, section=CRYPTO_PRICES_SECTION, provider=PROVIDER)
if crypto_prices.empty or "close" not in crypto_prices.columns:
    pd.DataFrame()
else:
    close = crypto_prices["close"]
    trend_signal = (close > close.rolling(100).mean()).astype(float)
    signal_backtest(close, trend_signal, annualization=365)

In [13]:
if crypto_prices.empty or "close" not in crypto_prices.columns:
    pd.DataFrame()
else:
    close = crypto_prices["close"]
    returns = close.pct_change()
    realized_vol = returns.rolling(30).std() * 365 ** 0.5
    risk_budget = pd.DataFrame({
        "close": close,
        "momentum_30d": close.pct_change(30),
        "momentum_120d": close.pct_change(120),
        "realized_vol_30d": realized_vol,
        "vol_target_weight_20pct_cap_1x": (0.20 / realized_vol).clip(0, 1),
        "drawdown": close / close.cummax() - 1,
    })
    preview(risk_budget, rows=15)

<!-- output-analysis -->
## Analysis Notes From This Run

For `BTCUSD`, the stored FMP series has very large long-run gains but also an 85.3% max drawdown and roughly 76.6% annualized volatility. The latest stored close is below the 200-day average by about 16.6%, so the current state in this warehouse snapshot is not a favorable long-only trend regime.

For a quant trader, the key takeaway is position sizing. A naive fixed-weight BTC sleeve can dominate portfolio drawdowns. The notebook's volatility-target and trend-filter diagnostics are the right first pass before testing any crypto allocation or breakout strategy.